In [ ]:
import math as m

class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector({self.x},{self.y})"

    def __str__(self):
        return f"({self.x},{self.y})"

    def mod2(self):
        return self.x**2 + self.y**2

    def mod(self):
        return m.sqrt(self.mod2())

    def __add__(self, other):
        return Vector(self.x+other.x, self.y + other.y)

    def __neg__(self):
        return Vector(-self.x, -self.y)

    def __sub__(self, other):
        return self + (-other)

    def __mul__(self, k):
        return Vector(k*self.x, k*self.y)

    __rmul__ = __mul__

    def __truediv__(self, k):
        return Vector(self.x/k, self.y/k)

    def dot(self, other):
        return self.x * other.x + self.y * other.y

    def cross(self, other):
        return self.x * other.y - self.y *  other.x

    def cross_z(self, z):
        return self.y*z - self.x*z


def sign(x):
    return int(x > 0) - int(x < 0)
    

In [ ]:
N = 100
ps = [Particle(1, 1, Vector(x, y), Vector(10*cos(a), 10*sin(a)))
      for x, y, a in zip(random.uniform(0, 50, N), random.uniform(0, 50, N), random.uniform(0, 2*pi, N))]
anim = animar_hist(ps)
HTML(anim.to_jshtml())

In [ ]:
def animar_hist(particulas, dt=0.05, frames=200, pasos_por_frame=5, bins=15):
    fig, (ax_p, ax_h) = plt.subplots(1, 2, figsize=(11, 5))

    ax_p.set_xlim(0, Particle.L); ax_p.set_ylim(0, Particle.L); ax_p.set_aspect("equal")
    circulos = [Circle((p.r.x, p.r.y), p.rho) for p in particulas]
    for c in circulos:
        ax_p.add_patch(c)

    v2_medio = np.mean([p.v.mod2() for p in particulas])
    sigma2 = v2_medio / 2
    v_max = 3 * np.sqrt(v2_medio)
    vv = np.linspace(0, v_max, 200)
    mb = vv / sigma2 * np.exp(-vv**2 / (2 * sigma2))

    def dibujar(t):
        for c, p in zip(circulos, particulas):
            c.center = (p.r.x, p.r.y)
        mods = [p.v.mod() for p in particulas]
        ax_h.clear()
        ax_h.hist(mods, bins=bins, range=(0, v_max), density=True, alpha=0.7)
        ax_h.plot(vv, mb, "r-", label="Maxwell-Boltzmann 2D")
        ax_h.set_xlim(0, v_max); ax_h.set_ylim(0, 1.5 * mb.max())
        ax_h.set_xlabel("|v|"); ax_h.set_ylabel("densidad")
        ax_h.set_title(f"t = {t:.1f}")
        ax_h.legend(loc="upper right")
        return circulos

    def init():
        return dibujar(0)                      # estado inicial, sin evolucionar

    def update(frame):
        for _ in range(pasos_por_frame):
            for p in particulas:
                p.evolve(dt)
            for i in range(len(particulas)):
                for j in range(i + 1, len(particulas)):
                    particulas[i] @ particulas[j]
        return dibujar((frame + 1) * pasos_por_frame * dt)

    anim = FuncAnimation(fig, update, frames=frames, init_func=init, interval=50, blit=False)
    plt.close(fig)
    return anim